In [ ]:
# Water Simulation Lab
This Colab notebook enables running molecular dynamics (MD) simulations of pure water and studying structure and dynamics.

MD simulations employ the TIP4P2005 water model.

Simulations are run using OpenMM [4] at the user-defined temperature.

### Usage
We recommend running MD simulations run on a single GPU. To enable GPU select `Runtime` from the menu, then `Change runtime type` and select `GPU`.

Note: Cells for preliminary operations should be executed one by one to prevent crashes.

### References

4. P. Eastman, J. Swails, J. D. Chodera et al. __OpenMM 7: Rapid development of high performance algorithms for molecular dynamics__ _PLoS Comput Biol._ 2017 13(7):e1005659 DOI: https://doi.org/10.1371/journal.pcbi.1005659

In [1]:
# @title 1. Set the environment for simulations and analyses
pip install openmm[cuda12]
pip install --upgrade MDAnalysis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 45.4 MB/s eta 0:00:00


In [ ]:
# @title 2. Configure your water system

import numpy as np
import os
import shutil
import ipywidgets as widgets
import warnings
import wget
import yaml

warnings.filterwarnings('ignore')

temperature = 293.15 #@param {type:"number"}
box_side_length = 2.10 #@param {type:"number"}
simulation_time = 2 #@param {type:"number"}
#@markdown <i>*Units: temperature [K], box side length [nm], simulation time [ns]<i>

system_name = f"{box_side_length:.2f}_{temperature:.2f}"

if not os.path.isdir(f"{name}"):
    os.system(f"mkdir -p {name}")
    os.system(f"mkdir -p {name}_tmp")
    
N_steps = simulation_time * 1000 / 0.002
    
config_sim_data = dict(temperature=float(temperature), box_side_length=box_side_length, N_steps=N_steps)
yaml.dump(config_sim_data, open(f'{name}/config_sim.yaml', 'w'))

In [ ]:
#@title <b><font color='#A79AB2'>MD simulation toolbox</font></b>

import time
import numpy as np
from fastprogress import progress_bar
from openmm import *
from openmm.app import *
from openmm.unit import *

def run_steps(simulation, steps, chunk=1000):
    starttime = time.time()
    nblocks, remainder = divmod(steps, chunk)

    for _ in progress_bar(range(nblocks)):
        simulation.step(chunk)

    if remainder:
        simulation.step(remainder)

    elapsed = time.time()-starttime
    print(f"Simulation time: {elapsed//3600:.0f}h {(elapsed//60)%60:.0f}min {elapsed%60:.2f}s")

def simulate(config):
    temperature = config['temperature']
    box_side_length = config['box_side_length']
    N_steps = config['N_steps']

    top = Topology()
    modeller = Modeller(top, [])

    ff = ForceField('charmm36.xml','charmm36/tip4p2005.xml')
    modeller.addSolvent(ff,model='tip4pew',boxSize=Vec3(box_side_length,box_side_length,box_side_length)*nanometer)

    n_atoms = modeller.topology.getNumAtoms()
    n_residues = modeller.topology.getNumResidues()
    n_waters = sum(res.name in {'HOH','WAT','TIP4'} for res in modeller.topology.residues())
    n_ions = n_residues-n_waters

    print(f"Number of atoms: {n_atoms}")
    print(f"Number of water molecules: {n_waters}")
    print(f"Number of ions: {n_ions}")
    print(f"Total molecules: {n_residues}")

    dt = 0.002*picoseconds
    Temperature = temperature*kelvin
    integrator = LangevinMiddleIntegrator(Temperature,1/picosecond,dt)

    system = ff.createSystem(modeller.topology,nonbondedMethod=PME,nonbondedCutoff=1*nanometer,constraints=HBonds)
    simulation = Simulation(modeller.topology,system,integrator)
    simulation.context.setPositions(modeller.positions)

    state = simulation.context.getState(getEnergy=True)
    e_0 = state.getPotentialEnergy()
    print(f"Initial potential energy: {e_0}")

    simulation.minimizeEnergy()

    state = simulation.context.getState(getEnergy=True,getPositions=True)
    e_1 = state.getPotentialEnergy()
    print(f"Potential energy after minimization: {e_1}")
    print(f"Energy change: {e_1-e_0}")

    with open('EM_output.pdb','w') as f:
        PDBFile.writeFile(simulation.topology,state.getPositions(),f)

    simulation.context.setVelocitiesToTemperature(Temperature)

    simulation.reporters.append(StateDataReporter('NVT_output.txt',1000,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    print("NVT")
    run_steps(simulation,50000,1000)
    simulation.reporters.pop()

    system.addForce(MonteCarloBarostat(1*bar,Temperature,25))
    simulation.context.reinitialize(preserveState=True)

    simulation.reporters.append(StateDataReporter('NPT_output.txt',1000,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    simulation.reporters.append(DCDReporter('NPT_output.dcd',5000))

    print("NPT")
    run_steps(simulation,N_steps,1000)

In [ ]:
# @title 3. Run MD simulation
config = yaml.safe_load(open(f'{name}/config_sim.yaml', 'r'))
simulate(config)

In [ ]:
#@title <b><font color='#A79AB2'>Analysis toolbox</font></b>
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
import MDAnalysis.analysis.rdf as RDF
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

def load_state_data(filename,columns):
    data = np.loadtxt(filename,delimiter=',')
    return {name:data[:,i] for i,name in enumerate(columns)}

def plot_timeseries(x,y,ylabel,title,average=False):
    plt.plot(x,y)

    if average:
        av = np.mean(y)
        print(f"Average {ylabel.lower()}: {av:.3f}")
        plt.axhline(av,color='tab:red',ls='--')

    plt.xlabel("Step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

def add_water_bonds(u,oxygen='O',hydrogens=('H1','H2')):
    bonds = []

    for res in u.residues:
        O = res.atoms.select_atoms(f"name {oxygen}")

        for hydrogen in hydrogens:
            H = res.atoms.select_atoms(f"name {hydrogen}")

            if len(O) == 1 and len(H) == 1:
                bonds.append((O[0].index,H[0].index))

    u.add_bonds(bonds)

In [ ]:
# @title 4. Load the trajectory
u = mda.Universe('EM_output.pdb','NPT_output.dcd')

print(f"Frames: {len(u.trajectory)}")
print(f"Atoms: {u.atoms.n_atoms}")
print(f"Water molecules: {len(u.select_atoms('resname HOH and name O'))}")

In [ ]:
# @title 5. NVT analysis
nvt = load_state_data("NVT_output.txt",['step','potential_energy','temperature','volume','density'])

plot_timeseries(nvt['step'],nvt['temperature'],"Temperature (K)","NVT temperature")
plot_timeseries(nvt['step'],nvt['density'],"Density","NVT density")

In [ ]:
# @title 6. NPT analysis
npt = load_state_data("NPT_output.txt",['step','potential_energy','temperature','density'])

plot_timeseries(npt['step'],npt['temperature'],"Temperature (K)","NPT temperature",average=True)
plot_timeseries(npt['step'],npt['density'],"Density","NPT density",average=True)

In [ ]:
# @title 7. Hydrogen-bond analysis
add_water_bonds(u)
print(f"Bonds added: {len(u.bonds)}")

hbonds = HBA(universe=u,hydrogens_sel='name H1 H2',donors_sel='name O',acceptors_sel='name O',update_selections=False,d_a_cutoff=3.5,d_h_a_angle_cutoff=140)
hbonds.run()

counts = hbonds.count_by_time()
n_waters = len(u.select_atoms("resname HOH and name O"))
hbonds_per_water = 2*counts/n_waters

trajectory_steps = np.arange(len(counts))*5000

plt.plot(trajectory_steps,hbonds_per_water)
plt.xlabel("Step")
plt.ylabel("Hydrogen bonds per water molecule")
plt.title("Hydrogen bonds")
plt.tight_layout()
plt.show()

print(f"Average hydrogen bonds per water molecule: {hbonds_per_water.mean():.2f}")

In [ ]:
# @title 8. Oxygen–oxygen radial distribution function
oxygen = u.select_atoms("resname HOH and name O")

rdf = RDF.InterRDF(oxygen,oxygen,range=(0.,9.),exclusion_block=(1,1),density=False)
rdf.run()

plt.plot(rdf.results.bins,rdf.results.rdf)
plt.xlabel(r"Distance ($\AA$)")
plt.ylabel(r"$g_{\mathrm{OO}}(r)$")
plt.title("Oxygen–oxygen RDF")
plt.grid()
plt.tight_layout()
plt.show()